# Compare Source vs Categorized Dataset Totals

This notebook compares `.wav` counts by `(dB, machine, label)` between:
1. recursive machine folders: `datasets/fan`, `datasets/slider`, `datasets/valve`
2. categorized folders like `datasets/-6db-fan-normal`

It prints both overall totals and clearly marks any mismatches.

In [1]:
from pathlib import Path
from collections import Counter
import re

TARGET_MACHINES = ('fan', 'slider', 'valve')
TARGET_LABELS = ('normal', 'abnormal')

FILE_RE = re.compile(r'(?P<db>[+-]?\d+)-(?P<machine>fan|slider|valve)-\d{2}-(?P<short>nm|ab)-\d+\.wav$', re.IGNORECASE)
CAT_DIR_RE = re.compile(r'^(?P<db>[+-]?\d+)db-(?P<machine>fan|slider|valve)-(?P<label>normal|abnormal)$', re.IGNORECASE)

def resolve_datasets_dir() -> Path:
    candidates = [Path('datasets'), Path('../datasets')]
    for candidate in candidates:
        if all((candidate / m).exists() for m in TARGET_MACHINES):
            return candidate
    raise FileNotFoundError('Could not find datasets directory from current working directory.')

DATASETS_DIR = resolve_datasets_dir()
print(f'Using datasets directory: {DATASETS_DIR.resolve()}')

def normalize_db(raw: str) -> str:
    n = int(raw)
    return f'{n:+d}db'

def parse_file_combo(path: Path):
    m = FILE_RE.search(path.name)
    if not m:
        return None
    db = normalize_db(m.group('db'))
    machine = m.group('machine').lower()
    label = 'normal' if m.group('short').lower() == 'nm' else 'abnormal'
    return (db, machine, label)

def db_sort_key(db: str):
    return int(db.replace('db', ''))

def combo_sort_key(combo):
    db, machine, label = combo
    return (TARGET_MACHINES.index(machine), db_sort_key(db), TARGET_LABELS.index(label))

source_counts = Counter()
unparsed_source_files = []

for machine in TARGET_MACHINES:
    machine_dir = DATASETS_DIR / machine
    if not machine_dir.exists():
        continue
    for wav_path in machine_dir.rglob('*.wav'):
        combo = parse_file_combo(wav_path)
        if combo is None:
            unparsed_source_files.append(str(wav_path))
            continue
        source_counts[combo] += 1

categorized_counts = Counter()
for child in DATASETS_DIR.iterdir():
    if not child.is_dir():
        continue
    m = CAT_DIR_RE.match(child.name)
    if not m:
        continue
    combo = (normalize_db(m.group('db')), m.group('machine').lower(), m.group('label').lower())
    categorized_counts[combo] += sum(1 for _ in child.rglob('*.wav'))

all_combos = sorted(set(source_counts) | set(categorized_counts), key=combo_sort_key)

print('Per-combination counts:')
print('-' * 88)
print(f"{'dB':<6} {'machine':<8} {'label':<9} {'source':>8} {'categorized':>12} {'diff':>8}  status")
print('-' * 88)
for combo in all_combos:
    s = source_counts.get(combo, 0)
    c = categorized_counts.get(combo, 0)
    diff = c - s
    status = 'MISMATCH <--' if diff != 0 else 'OK'
    db, machine, label = combo
    print(f'{db:<6} {machine:<8} {label:<9} {s:>8} {c:>12} {diff:>8}  {status}')

print('-' * 88)
source_total = sum(source_counts.values())
categorized_total = sum(categorized_counts.values())
total_diff = categorized_total - source_total
total_status = 'MISMATCH <--' if total_diff != 0 else 'OK'
print(f'Source total files      : {source_total}')
print(f'Categorized total files : {categorized_total}')
print(f'Total diff (cat - src)  : {total_diff}  {total_status}')

if unparsed_source_files:
    print('\nWARNING: Some source files were skipped because their names did not match the expected pattern.')
    print(f'Skipped files: {len(unparsed_source_files)}')
    for p in unparsed_source_files[:10]:
        print(f'  - {p}')
    if len(unparsed_source_files) > 10:
        print('  - ...')


Using datasets directory: /home/mitch/development/raccoon-ball/datasets
Per-combination counts:
----------------------------------------------------------------------------------------
dB     machine  label       source  categorized     diff  status
----------------------------------------------------------------------------------------
-6db   fan      normal        4075         4075        0  OK
-6db   fan      abnormal      1475         1475        0  OK
+0db   fan      normal        4075         4075        0  OK
+0db   fan      abnormal      1475         1475        0  OK
+6db   fan      normal        4075         4075        0  OK
+6db   fan      abnormal      1475         1475        0  OK
-6db   slider   normal        3204         3204        0  OK
-6db   slider   abnormal       890          890        0  OK
+0db   slider   normal        3204         3204        0  OK
+0db   slider   abnormal       890          890        0  OK
+6db   slider   normal        3204         3204    